# Notebook 01: Exploring LightRAG Graph & Index for Vietnamese Law

This notebook explores the entity/relation graph constructed by LightRAG on sample Vietnamese legal documents.

In [ ]:
import os
import sys
import json
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

# Add src to sys.path
sys.path.insert(0, os.path.abspath("../src"))
from utils import load_config, LightRAGConfig

## 1. Load LightRAG Config & Working Directory

In [ ]:
config = load_config("../config/lightrag_config.example.env")
working_dir = os.path.abspath(config.working_dir)
print(f"Working Directory: {working_dir}")

graphml_file = os.path.join(working_dir, "graph_chunk_entity_relation.graphml")
print(f"Graph file exists: {os.path.exists(graphml_file)}")

## 2. Load & Inspect NetworkX Graph Structure

In [ ]:
if os.path.exists(graphml_file):
    G = nx.read_graphml(graphml_file)
    print(f"Graph Statistics:")
    print(f"- Total Nodes (Entities): {G.number_of_nodes()}")
    print(f"- Total Edges (Relations): {G.number_of_edges()}")
    
    # Top connected entities
    degrees = dict(G.degree())
    top_entities = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:10]
    print("\nTop 10 Most Connected Legal Entities:")
    for node, deg in top_entities:
        print(f"  • {node}: {deg} connections")
else:
    print("Graph file not found. Run ingest.py first to build the index.")

## 3. Visualize Subgraph

In [ ]:
if os.path.exists(graphml_file) and G.number_of_nodes() > 0:
    plt.figure(figsize=(12, 8))
    # Sample top 20 nodes for clear visualization
    sample_nodes = [node for node, _ in top_entities[:20]]
    subG = G.subgraph(sample_nodes)
    
    pos = nx.spring_layout(subG, seed=42)
    nx.draw_networkx_nodes(subG, pos, node_size=700, node_color='skyblue')
    nx.draw_networkx_edges(subG, pos, alpha=0.5, edge_color='gray')
    nx.draw_networkx_labels(subG, pos, font_size=9, font_family='sans-serif')
    
    plt.title("LightRAG Legal Entity Subgraph Visualization", fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()